# AI Stock Prediction Training Pipeline
This notebook downloads 2 years of historical data from Yahoo Finance and trains a deep learning LSTM model.

In [ ]:
!pip install yfinance tensorflow scikit-learn pandas numpy matplotlib -q

## 1. Download 2 Years of Data as CSV

In [ ]:
import yfinance as yf
import pandas as pd

SYMBOL = 'RELIANCE.NS'
print(f'Downloading data for {SYMBOL}...')
ticker = yf.Ticker(SYMBOL)
df = ticker.history(period='2y')

# Save to CSV
csv_filename = f'{SYMBOL}_2y_historical.csv'
df.to_csv(csv_filename)
print(f'Saved 2 years of data to: {csv_filename}')
print(f'Shape: {df.shape}')
df.head()

## 2. Preprocess Data

In [ ]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

data = df['Close'].values.reshape(-1, 1)
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data)

SEQ_LEN = 30
X, y = [], []
for i in range(SEQ_LEN, len(scaled_data)):
    X.append(scaled_data[i-SEQ_LEN:i, 0])
    y.append(scaled_data[i, 0])

X, y = np.array(X), np.array(y)
X = np.reshape(X, (X.shape[0], X.shape[1], 1))

print('Training sequences constructed.')
print('X shape:', X.shape)
print('y shape:', y.shape)

## 3. Build & Train LSTM Neural Network

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential([
    LSTM(50, return_sequences=True, input_shape=(X.shape[1], 1)),
    Dropout(0.2),
    LSTM(50, return_sequences=False),
    Dropout(0.2),
    Dense(25),
    Dense(1)
])

model.compile(optimizer='adam', loss='mean_squared_error')

print('Training model...')
history = model.fit(X, y, batch_size=32, epochs=20, validation_split=0.1)

## 4. Evaluate Model Loss

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
# Save the trained model to h5 format
model.save('stock_lstm_model.h5')
print('Model saved successfully to stock_lstm_model.h5')